# 🎬 Reels — Free-GPU Talking Avatar (MuseTalk)

Generate a talking-avatar reel from a **script + one photo**, 100% free on a Colab **T4 GPU**.

**Runtime → Change runtime type → GPU (T4)** before running.

Pipeline: `script → edge-tts voice → MuseTalk lip-sync → 9:16 reel`.

> MuseTalk = fast face lip-sync (Apache-2.0). For hand/body motion swap in EchoMimicV2 (see repo README).


In [ ]:
# 1) Check the GPU (must show a Tesla T4 / similar)
!nvidia-smi -L
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2) Get this repo (scripts: tts.py, reel_utils.py, avatars/, examples/)
!git clone https://github.com/amsinghnavdeep/reels.git 2>/dev/null || (cd reels && git pull)
%cd /content/reels
!pip -q install edge-tts pydub pyyaml

In [ ]:
# 3) Clone + install MuseTalk (one-time, ~5-8 min)
import os
os.makedirs('engines', exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk
%cd /content/reels/engines/MuseTalk
!pip -q install -r requirements.txt
!pip -q install --no-cache-dir -U openmim
!mim install mmengine 'mmcv==2.0.1' 'mmdet==3.1.0' 'mmpose==1.1.0'
# static ffmpeg for MuseTalk
!mkdir -p ffmpeg && cd ffmpeg && wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz --strip-components=1
os.environ['FFMPEG_PATH'] = '/content/reels/engines/MuseTalk/ffmpeg'

In [ ]:
# 4) Download MuseTalk weights (~5 GB, one-time)
%cd /content/reels/engines/MuseTalk
!sh ./download_weights.sh || bash ./download_weights.sh

In [ ]:
# 5) Your script -> voice (edge-tts). Edit the script or upload your own photo.
%cd /content/reels
SCRIPT   = 'examples/script.txt'          # or upload your own .txt
AVATAR   = 'avatars/panditji.png'         # or upload your own front-facing photo
VOICE    = 'hi-IN-MadhurNeural'           # hi-IN-SwaraNeural (F), en-IN-PrabhatNeural ...
!python tts.py --script $SCRIPT --engine edge --voice $VOICE --output output/speech.wav
from IPython.display import Audio; Audio('output/speech.wav')

In [ ]:
# 6) Run MuseTalk lip-sync (image + audio -> talking video)
import yaml, os
os.chdir('/content/reels/engines/MuseTalk')
cfg = {'task_0': {'video_path': '/content/reels/'+AVATAR, 'audio_path': '/content/reels/output/speech.wav'}}
os.makedirs('configs/inference', exist_ok=True)
open('configs/inference/reels.yaml','w').write(yaml.dump(cfg))
!python -m scripts.inference \
    --inference_config configs/inference/reels.yaml \
    --result_dir /content/reels/output/muse \
    --version v15 --fps 25 --batch_size 4 --use_float16 \
    --parsing_mode jaw --extra_margin 10

In [ ]:
# 7) Compose a 9:16 reel + preview/download
%cd /content/reels
import glob
clips = sorted(glob.glob('output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
assert clips, 'No MuseTalk output found — check the previous cell logs.'
raw = clips[-1]; print('raw clip:', raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
from google.colab import files
from IPython.display import Video
files.download('output/reel.mp4')
Video('output/reel.mp4', embed=True, width=270)